# Visualizations for the three moe model in test_three_moe.py

In [ ]:
# Import statements
import matplotlib.pyplot as plt
import torch

from pypolymix.parameter_groups import DeterministicGroup, IIDGaussianGroup
from pypolymix.surrogate_models import PolynomialChaosExpansion, MixtureOfExperts, GatingNetwork
from pypolymix import StochasticModel

from train_moe import train_moe_model
from test_three_moe import make_three_region_data, make_three_expert_problem

In [ ]:
# Set constants and random seed
NUM_EXPERTS = 3

In [ ]:
# Generate synthetic data
X, Y, region, edges = make_three_region_data()

### The Synthetic Data

In [ ]:
_, ax = plt.subplots()
ax.scatter(X, Y)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()

In [ ]:
# Create stochastic three-expert model
surrogate_model, model, X, Y, region, edges = make_three_expert_problem()

print(f"Surrogate model has {surrogate_model.num_params()} parameters")
print(f"Created stochastic model with {model.num_params()} parameters")

In [ ]:
# Train the stochastic model
total_loss = train_moe_model(surrogate_model, model, X, Y)

print(f"Final total loss = {total_loss:.4f}")

In [ ]:
# Evaluate the model
model.eval()
with torch.no_grad():
    X_test = torch.linspace(-1, 1, 300).unsqueeze(1)
    Y_test = model(X_test, num_samples=1000)

### Data and Model

Plot the model's mean prediction on the same graph as the generated data.

Note that the 80th percentile prediction is also plotted, but the width is small enough that the uncertainty is not shown on the plot.

In [ ]:
# Plot prediction
_, ax = plt.subplots()
Q_test = torch.quantile(Y_test, torch.tensor([0.1, 0.9]), axis=0)
X_plot = X_test.squeeze(-1)
Q_plot = Q_test.squeeze(-1)
ax.fill_between(X_plot, Q_plot[0], Q_plot[-1], color="red", alpha=0.5, linewidth=0)
M_plot = Y_test.mean(axis=0).squeeze(-1)
ax.plot(X_plot, M_plot, color="red", linewidth=2, label="MoE mean")
ax.scatter(X, Y, zorder=99)
for edge in edges[1:-1]:
    ax.axvline(edge.item(), color="black", linestyle="--", linewidth=1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(frameon=False)
plt.tight_layout()